In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from helpers import make_preprocessor



In [6]:
df = pd.read_csv("data/ames_housing.csv")
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.drop(index=[523, 1298])
X_train = train_df.drop(columns =['Id', 'SalePrice', 'GarageYrBlt', 'PoolArea'])
y_train = train_df['SalePrice']

In [10]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

models = {
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1),
    "LightGBM": LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0)
}

rows = []

for name, model in models.items():
    pipe = Pipeline([('prep', make_preprocessor(X_train)), ('model', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
    rows.append({"Model": name, "MAE": -scores.mean(), "Spread": scores.std()})


pd.DataFrame(rows).sort_values("MAE").round(1)

,Model,MAE,Spread
2,CatBoost,14437.5,854.3
1,LightGBM,16164.8,982.3
0,XGBoost,17635.5,629.6


## Boosting libraries with default settings

Same cleaned training data, same preprocessing and same 5-fold split as before. None of these three is tuned.

| Model | MAE | Spread across folds |
|---|---|---|
| CatBoost | $14,438 | 854 |
| LightGBM | $16,165 | 982 |
| XGBoost | $17,636 | 630 |

For comparison: tuned Gradient Boosting $15,227, tuned Random Forest $16,583, Linear Regression baseline $17,207.

1. **CatBoost is ahead without any tuning.** It beats tuned Gradient Boosting by about $790, roughly one spread, so it is probably better but not certain.
2. **XGBoost is worse than the baseline.** Its default trees are deep and learn fast, which probably overfits on 1,166 houses. This explanation has not been tested.
3. **The comparison is not equal:** CatBoost, LightGBM and XGBoost are untuned, while Gradient Boosting and Random Forest were tuned. Next step: tune the three boosting libraries.